In [2]:
# # RAGU Score Analysis - Optimized Version

# ## How to Run
# 1. Set the `granularity` variable ('q' for quarterly, 'm' for monthly, 'w' for weekly)
# 2. Set `run_every_query` to True for fresh data, False to use cached pickles
# 3. Update `max_quarter`, `max_month` for the current period cutoff
# 4. Run All Cells

# ## Key Improvements Over Original
# - Removed unused code (auc_pred, float_to_vintage, distribution, t-1 table, waterfall exports)
# - Optimized wrapper loop with batch concatenation (O(n) instead of O(n^2))
# - Dynamic Excel export with configurable date ranges
# - Consolidated duplicate export logic into single function

In [3]:
# =============================================================================
# IMPORTS AND CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import copy
import openpyxl
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

# =============================================================================
# MAIN CONFIGURATION - UPDATE THESE VALUES EACH RUN
# =============================================================================
granularity = ['q', 'm', 'w'][1]  # 0=quarterly, 1=monthly, 2=weekly
run_every_query = True  # True to run SQL queries, False to use cached pickles
max_quarter, max_month = 1, 2  # Current period cutoffs for 2026
max_week = int(dt.date.today().strftime("%U"))

# Derived configuration (do not modify)
use_app_date_vintage = (granularity == 'w')
date_col = 'app_date' if use_app_date_vintage else 'book_date'
min_date = "'2025-01-01'" if use_app_date_vintage else "'2016-01-01'"
start_year = 2025 if use_app_date_vintage else 2016

# =============================================================================
# DATE RANGE CONFIGURATION (for dynamic Excel export)
# =============================================================================
DATE_CONFIG = {
    'start_year': start_year,
    'end_year': 2027,
    'yearly_aggregation_years': list(range(2016, 2020)),  # Years to aggregate into yearly totals
    'recent_vintage_cutoff': 2020,  # Show individual periods from this year onward
    'current_year': 2026,
    'max_quarter': max_quarter,
    'max_month': max_month,
    'max_week': max_week,
}

# Excel cell positions for each LOB
EXCEL_CELL_CONFIG = {
    'quarterly_monthly': {
        'AN': 'B2', 'FRN': 'B20', 'STG': 'B29', 'non_kmxent': 'B47',
        'ENT': 'F38', 'FLD': 'F11', 'KMX': 'B56', 'POS': 'B65'
    },
    'weekly': {
        'AN': 'B2', 'FRN': 'B20', 'STG': 'B29', 'non_kmxent': 'B47',
        'ENT': 'B38', 'FLD': 'B11', 'KMX': 'B56', 'POS': 'B65'
    }
}

# Baseline parameters for RAGU calculations
BASELINE_CONFIG = {
    'non_kmxent': {'ltv': 1.94, 'recovery_unadjusted': 0.4308, 'new_recovery_unadjusted': 0.55, 'recovery_100_ltv': 0.4308, 'recovery': 0.4308},
    'ENT': {'ltv': 1.45, 'recovery_unadjusted': 0.46, 'new_recovery_unadjusted': 0.54, 'recovery_100_ltv': 0.456, 'recovery': 0.456},
    'KMX': {'ltv': 1.5863, 'recovery_unadjusted': 0.3546, 'new_recovery_unadjusted': 0.58, 'recovery_100_ltv': 0.3546, 'recovery': 0.3546}
}

# Model parameters
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'ltv_realization_dollar': 0
}

In [4]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def vintage_to_float(vintage):
    """
    Converts a vintage of form 'YYYY QQ' or 'YYYY MMM' to a float.
    Examples: '2016 Q2' -> 2016.25, '2016 M06' -> 2016.4167
    """
    if vintage[5] == 'Q':
        return int(vintage[:4]) + (int(vintage[-1]) - 1) / 4
    elif vintage[5] == 'M':
        return int(vintage[:4]) + (int(vintage[-2:]) - 1) / 12
    else:
        # Weekly format: 'YYYY-WW'
        return int(vintage[:4]) + int(vintage[-2:]) / 52


def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    """
    Run a SQL Query by reading from a .txt file or using direct query string.
    
    Parameters:
        filename (str): File path to read or direct query string if filename_is_query=True
        sub_list (list): List of (placeholder, value) tuples for substitution
        connection: Optional existing database connection
        filename_is_query (bool): If True, treat filename as a query string
    
    Returns:
        pd.DataFrame: Query results
    """
    if sub_list is None:
        sub_list = []
    
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    
    for text, var in sub_list:
        query = query.replace(text, var)
    
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    """Stores data to a pickle file. Handles swapped arguments gracefully."""
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    """Loads data from a pickle file."""
    with open(filename, 'rb') as file:
        return pickle.load(file)


def smooth(series):
    """Calculates a 5-point centered moving average."""
    averaged_series = pd.Series(index=series.index, dtype=float)
    
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    """Returns the weighted average of a metric by proceeds (amount financed)."""
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    """
    Computes weighted averages of metrics by amt_financed.
    Returns a Series with weighted averages and total amt_financed.
    """
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)

In [5]:
# =============================================================================
# ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    """
    Calculate Unit Loss Adjustment multiplier for Non-KMX LOBs.
    Applies various risk factors to adjust the loss probability.
    """
    ula_df['loss_multiplier'] = 1

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=0.067, leave_out='None'):
    """
    Calculate Unit Loss Adjustment multiplier for KMX LOB.
    Handles Mountain 3.0, 3.1, 3.2, and 4.1 model versions.
    """
    ula_df['loss_multiplier'] = 1

    # Low FICO (MTN 3.0)
    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))

    # Low Vantage (MTN 3.0)
    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))

    # High PTI (MTN 3.0)
    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)

    # Loss scale division (MTN 3.0)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)

    # Secured credit (ALL MTN)
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))

    # Authorized tradelines (MTN 3.0)
    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    # Soft pull (MTN 3.0)
    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    # Fraud (ALL MTN)
    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    # Clip (MTN 3.0 non-tier3)
    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))

    # Vehicle Age (MTN 3.0)
    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    # NPC flag
    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    # Student Loans (ALL MTN)
    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag

    # High sales price (MTN 4.1)
    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    # Driver flag (ALL MTN)
    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    # Louisiana (ALL MTN)
    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag

    # Georgia (ALL MTN)
    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag

    # TX/CA (ALL MTN)
    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag

    # State counter adjustment (ALL MTN)
    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012

    # Chime (ALL MTN)
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)

    # Job time (ALL MTN)
    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    # Existing DQ (ALL MTN)
    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    # Employment type (ALL MTN)
    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    # Authorized tradelines (MTN 3.1, 3.2, 4.1)
    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    # Soft pull (MTN 3.1, 3.2, 4.1)
    if leave_out != 'Soft pull':
        mask_soft = ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag
        mask_hard = ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag
        ula_df.loc[mask_soft, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.97 + 0.15 * ula_df.cd_perc_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[mask_hard, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.954 + 0.346 * ula_df.cd_perc_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

    # Final clip (MTN 3.1, 3.2, 4.1)
    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

In [6]:
# =============================================================================
# CORE RAGU SCORE FUNCTION
# =============================================================================

def get_ragu_score(vintage, lob, lob_type, ula_df_total, rra_df_total, new_recovery, 
                   ms_df, baseline_config, model_params, leave_out='None'):
    """
    Core RAGU Score calculation for a single vintage and LOB.
    
    This function:
    1. Applies Unit Loss Adjustments (ULA)
    2. Merges with Recovery Rate data
    3. Computes weighted averages and RAGU score formulas
    
    Parameters:
        vintage (str): Vintage string (e.g., '2022 Q1' or '2022 M03')
        lob (str|tuple): LOB or tuple of LOBs
        lob_type (str): 'non_kmxent', 'ENT', or 'KMX'
        ula_df_total (DataFrame): All ULA data
        rra_df_total (DataFrame): All RRA data
        new_recovery (DataFrame): Recovery multipliers
        ms_df (DataFrame): Model scores by vintage/LOB
        baseline_config (dict): Baseline parameters for this lob_type
        model_params (dict): Model parameters (mean_unit_loss, etc.)
        leave_out (str): Factor to exclude for leave-out analysis
    
    Returns:
        DataFrame: RAGU results with all metrics
    """
    # Extract baseline parameters
    baseline_ltv = baseline_config['ltv']
    baseline_recovery_pct = baseline_config['recovery']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_recovery_100_ltv_pct = baseline_config['recovery_100_ltv']
    
    # Extract model parameters
    mean_unit_loss = model_params['mean_unit_loss']
    unit_loss_to_model_score = model_params['unit_loss_to_model_score']
    
    # Filter ULA data
    if isinstance(lob, str):
        ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob == lob)].copy()
    else:
        ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob.isin(lob))].copy()

    # Apply ULA multipliers
    if lob == 'KMX':
        ula_df = get_ula_multiplier_kmx(ula_df, loss_scale=0.067, leave_out=leave_out)
    else:
        ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)
    
    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier']]

    # Get Recovery Rate data
    if leave_out == 'None':
        if isinstance(lob, str):
            rra_df = rra_df_total[(rra_df_total.vintage == vintage) & (rra_df_total.lob == lob)].copy()
        else:
            rra_df = rra_df_total[(rra_df_total.vintage == vintage) & (rra_df_total.lob.isin(lob))].copy()
    else:
        if isinstance(lob, str):
            rra_df = rra_df_total[(rra_df_total.lob == lob)].copy()
        else:
            rra_df = rra_df_total[(rra_df_total.lob.isin(lob))].copy()

    try:
        rra_df.car_year = rra_df.car_year.fillna(round(rra_df.car_year.mean()))
    except:
        store_pickle('rra_df_pickle_exception', rra_df)

    rra_df['car_age_orig'] = np.maximum(
        (pd.to_datetime(rra_df[date_col]) - pd.to_datetime(rra_df.car_year.astype(int).astype(str) + '-09-01')).dt.days / 365, 
        1 / 365)

    # Merge with new recovery
    rra_df = rra_df.merge(new_recovery[['account_number', 'new_recovery_multiplier']], on='account_number', how='left')

    if leave_out == 'None':
        rra_df['recovery_multiplier'] = rra_df['new_recovery_multiplier']
    else:
        rra_df = rra_df[rra_df.vintage == vintage]
        rra_df['recovery_multiplier'] = rra_df['new_recovery_multiplier']

    rra_df = rra_df[['account_number', 'recovery_multiplier']]
    
    # Merge ULA and RRA
    mix_df = ula_df.merge(rra_df, on='account_number', how='inner').drop_duplicates(subset='account_number', keep='first')
    bb_populated_df = mix_df.dropna(subset='bbvalue')
    bb_populated_df['ltv'] = bb_populated_df.amt_financed / bb_populated_df.bbvalue
    bb_populated_df['recovery_unadjusted_multiplier'] = rra_df['recovery_multiplier']

    # Group by LOB
    grouped_mix_df = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum, 
        ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue'],
        include_groups=False
    )

    # Calculate recovery metrics
    grouped_mix_df['ltv_realization_factor'] = 0
    grouped_mix_df['recovery_100_ltv_multiplier'] = 0
    grouped_mix_df['recovery_multiplier'] = 0

    # Merge with model scores
    vintage_ms_df = ms_df[ms_df.vintage == vintage].copy()
    
    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    # Calculate RAGU metrics
    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')
    
    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_recovery'] = 0
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()
    full_df['baselined_100_ltv_recovery'] = 0

    # Handle multiple LOBs (non_kmxent case)
    if not isinstance(lob, str) and len(lob) > 1:
        full_df.loc[lob_type, 'unit_loss_score'] = weight_by_proceeds(full_df.unit_loss_score, full_df.amt_financed_x)
        full_df.loc[lob_type, 'est_unit_loss'] = mean_unit_loss
        full_df.loc[lob_type, 'ms_original'] = weight_by_proceeds(full_df.model_score, full_df.amt_financed_x)
        full_df.loc[lob_type, 'baselined_recovery'] = weight_by_proceeds(full_df.recovery_multiplier / baseline_recovery_pct, full_df.amt_financed_x)
        full_df.loc[lob_type, 'baselined_unadjusted_recovery'] = weight_by_proceeds(full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct, full_df.amt_financed_x)
        full_df.loc[lob_type, 'baselined_100_ltv_recovery'] = weight_by_proceeds(full_df.recovery_100_ltv_multiplier / baseline_recovery_100_ltv_pct, full_df.amt_financed_x)
        full_df.loc[lob_type, 'recovery_multiplier'] = weight_by_proceeds(full_df.recovery_multiplier, full_df.amt_financed_x)
        full_df.loc[lob_type, 'recovery_unadjusted_multiplier'] = weight_by_proceeds(full_df.recovery_unadjusted_multiplier, full_df.amt_financed_x)
        full_df.loc[lob_type, 'recovery_100_ltv_multiplier'] = weight_by_proceeds(full_df.recovery_100_ltv_multiplier, full_df.amt_financed_x)
        full_df.loc[lob_type, 'vintage'] = vintage
        full_df.loc[lob_type, 'ltv'] = weight_by_proceeds(full_df.ltv, full_df.amt_financed_x)
        full_df.loc[lob_type, 'amt_financed_x'] = full_df.amt_financed_x.sum()

    # Final RAGU calculations
    full_df['ms_gla'] = full_df.unit_loss_score.copy()
    full_df['ms_exclude_ltv'] = ((1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score 
                                 + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery)
    full_df['only_recovery_ragu'] = full_df['ms_exclude_ltv'] - full_df['ms_gla']
    full_df['ms_100_ltv'] = ((1 - full_df.est_unit_loss * full_df.recovery_100_ltv_multiplier) * full_df.unit_loss_score 
                             + full_df.est_unit_loss * full_df.recovery_100_ltv_multiplier * full_df.unit_loss_score * full_df.baselined_100_ltv_recovery)
    full_df['ragu_score'] = ((1 - full_df.est_unit_loss * full_df.recovery_multiplier) * full_df.unit_loss_score 
                             + full_df.est_unit_loss * full_df.recovery_multiplier * full_df.unit_loss_score * full_df.baselined_recovery)

    return full_df

In [7]:
# =============================================================================
# FETCH MODEL SCORES
# =============================================================================

run_query = False
if run_query or run_every_query:
    all_original_model_scores = run_sql('postmodern_ms_query.txt', sub_list=[('{min_book_date}', min_date)])
    store_pickle(all_original_model_scores, 'all_original_model_scores_pickle')
else:
    all_original_model_scores = get_pickle('all_original_model_scores_pickle')

all_original_model_scores[date_col] = all_original_model_scores[date_col].astype(str)
all_original_model_scores.book_week = all_original_model_scores.book_week.astype(str)
all_original_model_scores.app_week = all_original_model_scores.app_week.astype(str)

# Assign vintages based on granularity
if granularity == 'q':
    all_original_model_scores['vintage'] = (all_original_model_scores[date_col].str[:4] + ' Q' + 
                                            ((all_original_model_scores[date_col].str[5:7].astype(int) - 1) // 3 + 1).astype(str))
elif granularity == 'm':
    all_original_model_scores['vintage'] = (all_original_model_scores[date_col].str[:4] + ' M' + 
                                            all_original_model_scores[date_col].str[5:7])
else:
    all_original_model_scores['vintage'] = (all_original_model_scores[date_col].str[:4] + '-' + 
                                            all_original_model_scores.app_week.str.zfill(2))

# Aggregate model scores
original_model_scores = all_original_model_scores.groupby(['vintage', 'lob']).apply(
    weighted_average_and_sum, 'model_score', include_groups=False).reset_index()
original_model_scores = pd.concat([
    original_model_scores, 
    all_original_model_scores.groupby(['vintage']).apply(
        weighted_average_and_sum, 'model_score', include_groups=False).reset_index()
]).fillna('POS')
original_model_scores = pd.concat([
    original_model_scores, 
    all_original_model_scores[all_original_model_scores.lob.isin(['AN', 'STG', 'FRN', 'FLD'])].groupby(['vintage']).apply(
        weighted_average_and_sum, 'model_score', include_groups=False).reset_index()
]).fillna('non_kmxent')

ms_df = original_model_scores.copy()
print(f"Loaded model scores: {len(ms_df)} records")
ms_df.head()

Loaded model scores: 963 records


,vintage,lob,model_score,amt_financed
0,2016 M01,AN,129.760371,7781530.53
1,2016 M01,FLD,136.535021,76648.00
2,2016 M01,FRN,128.589400,3254221.42
3,2016 M01,KMX,133.360584,918367.86
4,2016 M01,STG,129.003442,8187381.84


In [8]:
# =============================================================================
# FETCH ULA, RRA, AND RECOVERY DATA
# =============================================================================

expected_years_on_book = MODEL_PARAMS['expected_years_on_book']
impound_probability = MODEL_PARAMS['impound_probability']
mmi_standard_increase = MODEL_PARAMS['mmi_standard_increase']

run_query = False
generate_dfs = True

if generate_dfs:
    if run_query or run_every_query:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            print('temptables finished')
            
            ula_df_total = run_sql('vintage_level_ula_query.txt', sub_list=[('{min_book_date}', min_date)], connection=conn)
            print('ula query finished')

            rra_df_total = run_sql('vintage_level_rra_query.txt', sub_list=[('{min_book_date}', min_date)], connection=conn)
            dla_df = run_sql('new_dll_query.txt', connection=conn)
            print('tables query finished')

            new_recovery = run_sql('new_recovery_queryt.txt', connection=conn)
            print('new recovery finished')

        store_pickle((ula_df_total, rra_df_total, dla_df), '(ula_df_total, rra_df_total, dla_df)_unrefined_pickle')
    else:
        ula_df_total, rra_df_total, dla_df = get_pickle('(ula_df_total, rra_df_total, dla_df)_unrefined_pickle')
    
    # Filter out Core LOB (missing model scores)
    rra_df_total = rra_df_total[rra_df_total.lob != 'Core']
    ula_df_total = ula_df_total[ula_df_total.lob != 'Core']
    
    # MTN 3.1 flag
    ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
    ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'

    # Convert date columns
    rra_df_total[date_col] = rra_df_total[date_col].astype(str)
    new_recovery[date_col] = new_recovery[date_col].astype(str)
    ula_df_total[date_col] = ula_df_total[date_col].astype(str)

    rra_df_total.book_week = rra_df_total.book_week.astype(str)
    new_recovery.book_week = new_recovery.book_week.astype(str)
    ula_df_total.book_week = ula_df_total.book_week.astype(str)
    rra_df_total.app_week = rra_df_total.app_week.astype(str)
    new_recovery.app_week = new_recovery.app_week.astype(str)
    ula_df_total.app_week = ula_df_total.app_week.astype(str)

    # Vehicle age
    ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col].str[:4].astype(int) - ula_df_total.model_year, 1/365)
    ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
    ula_df_total.make = ula_df_total.make.str.upper().str[:3]
    ula_df_total.lob_or_bucket = np.select(
        [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
         ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])], 
        ['D', 'C'], default=ula_df_total.lob_or_bucket)
    ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col].str[:4].astype(int) + 
                                              ula_df_total[date_col].str[5:7].astype(int)/12 - 
                                              (ula_df_total.model_year - 0.25) - 1)
   
    # Dealer Level Loss (DLL) merge
    dla_df = dla_df.rename(columns={"valid_vintage": "book_vintage"})
    ula_df_total = ula_df_total.replace("2026 Q1", "current")
    ula_df_total = pd.merge(ula_df_total, dla_df, how="left", on=['dealer_number', 'book_vintage'])
    ula_df_total = ula_df_total.replace("current", "2026 Q1")
    ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
    ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
    ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

    # RRA data processing
    rra_df_total['yob'] = expected_years_on_book
    rra_df_total['impound_prob'] = impound_probability
    rra_df_total.car_make = rra_df_total.car_make.str.upper()
    rra_df_total['car_class_only'] = np.select(
        [rra_df_total.vehicle_class.str.contains('Pickup', na=False),
         rra_df_total.vehicle_class.str.contains('Sporty', na=False),
         rra_df_total.vehicle_class.str.contains('Small.*Car', na=False),
         rra_df_total.vehicle_class.str.contains('Car', na=False),
         rra_df_total.vehicle_class.str.contains('Small.*SUV', na=False),
         rra_df_total.vehicle_class.str.contains('Large.*SUV', na=False),
         rra_df_total.vehicle_class.str.contains('Minivan', na=False),
         rra_df_total.vehicle_class.str.contains(' Van', na=False)],
        ['Truck', 'Sporty', 'Compact', 'Car', 'SUV Small', 'SUV Large', 'Minivan', 'Work Van'],
        default='Other')
    rra_df_total.fuel_type = np.select(
        [rra_df_total.fuel_type.str.contains('Electric|Plug|EV', na=False),
         rra_df_total.fuel_type.str.contains('Hyb', na=False),
         ((rra_df_total.fuel_type == 'CNG') | (rra_df_total.fuel_type == 'LPG')),
         ((rra_df_total.fuel_type == 'null') | (rra_df_total.fuel_type == None))],
        ['EV', 'Hybrid', 'Gas', 'Gas'],
        default=rra_df_total.fuel_type)
    rra_df_total['car_lux'] = np.where(rra_df_total.vehicle_class.str.contains('Luxury', na=False), 'Luxury', 'Standard')
    rra_df_total['car_japanese'] = np.where(
        rra_df_total.car_make.isin(['HONDA', 'ACURA', 'ISUZU', 'MAZDA', 'MITSUBISHI', 'SUZUKI', 'TOYOTA', 'LEXUS', 'SCION', 'TOYOYA']),
        'Japanese', 'Other')
    
    warnings.filterwarnings("ignore", category=UserWarning)
    rra_df_total.job_company = rra_df_total.job_company.fillna('not provided')
    rra_df_total['driver_flag'] = np.where(
        rra_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
        1, 0)
    warnings.filterwarnings("default", category=UserWarning)
    
    # Add driver flag to ULA
    ula_df_total = ula_df_total.merge(rra_df_total[['account_number', 'driver_flag']], on='account_number', how='left')
    
    rra_df_total['mileage'] = rra_df_total.mileage_orig / 1000
    rra_df_total['r_mmi'] = mmi_standard_increase ** rra_df_total.yob
    
    # Handle NA values in ULA
    ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
    ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
    ula_df_total.employment = ula_df_total.employment.fillna('not seasonal or waiter')
    ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
    ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)
    
    # Populate flags
    ula_df_total['dealer_filter_a'] = ula_df_total.lob.isin({'AN', 'STG', 'FRN', 'MCY', 'FLD', 'ENT'})
    ula_df_total['dealer_filter_b'] = ula_df_total.lob.isin({'AN', 'STG', 'FRN', 'FLD', 'ENT'})
    
    # NonKMX Flags
    ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
    ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
    ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
    ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
    ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
    ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
    ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
    ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col] >= '2022-07-01')
    ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
    ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
    ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
    ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
    ula_df_total['chime_flag'] = ula_df_total.chime_indicator
    ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
    ula_df_total['seasonal_employment_flag'] = ula_df_total.employment == 'seasonal'
    ula_df_total['waiter_employment_flag'] = ula_df_total.employment == 'waiter'
    ula_df_total['high_cash_down_echopark_flag'] = False
    ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
    ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
    ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
    ula_df_total['null_fico_null_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & ((ula_df_total.vantage_score < 300) | (ula_df_total.vantage_score > 850))
    ula_df_total['pricing_change_flag'] = ula_df_total[date_col] >= '2024-10-01'
    ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
    ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col] >= '2023-05-01'
    ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
    ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

    # KMX Flags
    ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
    ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
    ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
    ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
    ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
    ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
    ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
    ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
    ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
    ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
    ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
    ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
    ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
    ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
    ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
    ula_df_total['kmx_toyho_flag'] = ula_df_total.cd_model_score >= 130 & ula_df_total.make.isin({'HON', 'TOY'})
    ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
    ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | (ula_df_total.vantage_score > 4) & ((ula_df_total.vantage_score < 450))
    ula_df_total['soft_pull_flag'] = ula_df_total.pull_type == 'softpull'
    ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
    ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
    ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

    # Add vintages
    if granularity == 'q':
        rra_df_total['vintage'] = rra_df_total[date_col].str[:4] + ' Q' + ((rra_df_total[date_col].str[5:7].astype(int) - 1) // 3 + 1).astype(str)
        ula_df_total['vintage'] = ula_df_total[date_col].str[:4] + ' Q' + ((ula_df_total[date_col].str[5:7].astype(int) - 1) // 3 + 1).astype(str)
        new_recovery['vintage'] = new_recovery[date_col].str[:4] + ' Q' + ((new_recovery[date_col].str[5:7].astype(int) - 1) // 3 + 1).astype(str)
    elif granularity == 'm':
        rra_df_total['vintage'] = rra_df_total[date_col].str[:4] + ' M' + rra_df_total[date_col].str[5:7]
        ula_df_total['vintage'] = ula_df_total[date_col].str[:4] + ' M' + ula_df_total[date_col].str[5:7]
        new_recovery['vintage'] = new_recovery[date_col].str[:4] + ' M' + new_recovery[date_col].str[5:7]
    elif granularity == 'w':
        rra_df_total['vintage'] = rra_df_total[date_col].str[:4] + '-' + rra_df_total.app_week.str.zfill(2)
        ula_df_total['vintage'] = ula_df_total[date_col].str[:4] + '-' + ula_df_total.app_week.str.zfill(2)
        new_recovery['vintage'] = new_recovery[date_col].str[:4] + '-' + new_recovery.app_week.str.zfill(2)

    # Handle duplicate employment records
    ula_df_total = ula_df_total.drop_duplicates().copy()
    ula_df_total['employment_type_code'] = ula_df_total.seasonal_employment_flag * 1 + ula_df_total.waiter_employment_flag * 8
    combined_employment_code_df = ula_df_total.groupby('account_number').employment_type_code.sum().reset_index()
    ula_df_total = ula_df_total.drop(columns='employment_type_code').merge(combined_employment_code_df, on='account_number')
    ula_df_total.seasonal_employment_flag = (ula_df_total.employment_type_code == 1) | (ula_df_total.employment_type_code == 9)
    ula_df_total.waiter_employment_flag = (ula_df_total.employment_type_code == 8) | (ula_df_total.employment_type_code == 9)
    ula_df_total = ula_df_total.drop(columns='employment').drop_duplicates()
    
    # Handle duplicate driver flags in RRA
    combined_driver_flag_df = rra_df_total.groupby('account_number').driver_flag.max().reset_index()
    rra_df_total = rra_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
    rra_df_total = rra_df_total.drop(columns='job_company').drop_duplicates()
    
    store_pickle((ula_df_total, rra_df_total), '(ula_df_total, rra_df_total)_refined_pickle')
else:
    ula_df_total, rra_df_total = get_pickle('(ula_df_total, rra_df_total)_refined_pickle')

print(f"ULA records: {len(ula_df_total):,}")
print(f"RRA records: {len(rra_df_total):,}")

temptables finished
ula query finished
tables query finished
new recovery finished
ULA records: 955,664
RRA records: 954,880


In [9]:
# =============================================================================
# OPTIMIZED WRAPPER - RAGU SCORE CALCULATIONS
# =============================================================================
# Key optimization: Collect results in lists and concatenate once at the end
# instead of sequential pd.concat which has O(n^2) memory usage

leave_out_list = ['None']  # For leave-out analysis, uncomment full list below
# leave_out_list = ['None', 'Previous ACA chargeoff', 'Small amount financed', 'Zero cash down', 
#                   'High mileage vehicle', 'High PTI', 'Car make', 'Theft risk', 
#                   'MCY high model score, low mileage', 'Weekday/weekend decision',
#                   'Secured credit (Chime, etc.)', 'Employment type', 'Authorized tradelines',
#                   'Clip', 'Vehicle Age', 'Dealer Level (Non-KMX)', 'Driver flag', 
#                   'Student Loans', 'Low PTI', 'Fraud']

# Excluded vintages (data quality issues)
EXCLUDED_VINTAGES = {
    'KMX': {'2022 M02', '2023 M11', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2023-44', '2023-45', '2023-46', '2023-47', '2023-48'},
    'non_kmxent': {'2023-14', '2023-15'},
    'ENT': {'2023-14', '2023-15'}
}

def generate_vintages(start_year, end_year, granularity, max_period):
    """Generate list of vintage strings based on granularity."""
    vintages = []
    for year in range(start_year, end_year):
        if granularity == 'q':
            periods = range(1, 5)
            max_p = max_period if year == DATE_CONFIG['current_year'] else 4
        elif granularity == 'm':
            periods = range(1, 13)
            max_p = max_period if year == DATE_CONFIG['current_year'] else 12
        else:
            periods = range(1, 53)
            max_p = max_period if year == DATE_CONFIG['current_year'] else 52
        
        for p in periods:
            if year == DATE_CONFIG['current_year'] and p > max_p:
                continue
            if granularity == 'q':
                vintages.append(f"{year} Q{p}")
            elif granularity == 'm':
                vintages.append(f"{year} M{str(p).zfill(2)}")
            else:
                vintages.append(f"{year}-{str(p).zfill(2)}")
    return vintages

def run_ragu_wrapper():
    """
    Main wrapper for RAGU Score calculations.
    Returns all_df, all_df_no_dropout, and full_df_list_dict_dict.
    """
    all_df_no_dropout = None
    full_df_list_dict_dict = {}
    
    # Get available vintages from model scores
    available_vintages_dict = {
        'non_kmxent': list(ms_df[ms_df.lob.isin(['FRN', 'STG', 'AN', 'FLD'])].vintage.unique()),
        'ENT': list(ms_df[ms_df.lob == 'ENT'].vintage.unique()),
        'KMX': list(ms_df[ms_df.lob == 'KMX'].vintage.unique())
    }
    
    # LOB configurations
    lob_configs = [
        (('FRN', 'AN', 'STG', 'FLD'), 'non_kmxent'),
        ('ENT', 'ENT'),
        ('KMX', 'KMX')
    ]
    
    for leave_out in leave_out_list:
        full_df_list_dict = {'non_kmxent': [], 'ENT': [], 'KMX': []}
        vintages_dict = {'non_kmxent': [], 'ENT': [], 'KMX': []}
        
        for lob, lob_type in lob_configs:
            available_vintages = available_vintages_dict[lob_type]
            baseline_config = BASELINE_CONFIG[lob_type]
            excluded = EXCLUDED_VINTAGES.get(lob_type, set())
            
            # Generate vintages for this LOB
            max_p = DATE_CONFIG['max_quarter'] if granularity == 'q' else DATE_CONFIG['max_month'] if granularity == 'm' else DATE_CONFIG['max_week']
            all_vintages = generate_vintages(start_year, DATE_CONFIG['end_year'], granularity, max_p)
            
            # Process each vintage
            for vintage in all_vintages:
                if vintage in vintages_dict[lob_type]:
                    continue
                if vintage not in available_vintages:
                    continue
                if vintage in excluded:
                    continue
                
                print(vintage, lob)
                full_df = get_ragu_score(
                    vintage, lob, lob_type, ula_df_total, rra_df_total, new_recovery,
                    ms_df, baseline_config, MODEL_PARAMS, leave_out=leave_out
                )
                
                full_df_list_dict[lob_type].append(full_df)
                vintages_dict[lob_type].append(vintage)
        
        # OPTIMIZED: Batch concatenation instead of sequential
        dfs_to_concat = []
        for lob_type in ['non_kmxent', 'ENT', 'KMX']:
            dfs_to_concat.extend(full_df_list_dict[lob_type])
        
        if dfs_to_concat:
            all_df = pd.concat(dfs_to_concat)
        else:
            all_df = pd.DataFrame()
        
        # Add POS (Portfolio Summary)
        if len(all_df.reset_index(names='lob').lob.unique()) > 1:
            individual_lob_df = all_df.reset_index(names='lob').sort_values(['vintage', 'lob']).reset_index(drop=True)
            individual_lob_df = individual_lob_df[individual_lob_df.lob == individual_lob_df.lob.str.upper()].rename(columns={'amt_financed_x': 'amt_financed'})
            pos_df = individual_lob_df.groupby('vintage').apply(
                weighted_average_and_sum, 
                ['ltv', 'ms_original', 'ms_gla', 'ms_exclude_ltv', 'ms_100_ltv', 'ragu_score'],
                include_groups=False
            ).reset_index()
            pos_df['lob'] = 'POS'
            pos_df = pos_df.rename(columns={'amt_financed': 'amt_financed_x'})
            all_df = pd.concat([all_df, pos_df.set_index('lob')])
        
        if leave_out == 'None':
            all_df_no_dropout = all_df.copy()
        
        full_df_list_dict_dict[leave_out] = full_df_list_dict
    
    return all_df, all_df_no_dropout, full_df_list_dict_dict, vintages_dict

# Run the wrapper
print("Starting RAGU calculations...")
start_time = time.time()
all_df, all_df_no_dropout, full_df_list_dict_dict, vintages_dict = run_ragu_wrapper()
elapsed = time.time() - start_time
print(f"\nRAGU calculations completed in {elapsed:.1f} seconds")
print(f"Total records in all_df: {len(all_df):,}")

Starting RAGU calculations...
2016 M01 ('FRN', 'AN', 'STG', 'FLD')
2016 M02 ('FRN', 'AN', 'STG', 'FLD')
2016 M03 ('FRN', 'AN', 'STG', 'FLD')
2016 M04 ('FRN', 'AN', 'STG', 'FLD')
2016 M05 ('FRN', 'AN', 'STG', 'FLD')
2016 M06 ('FRN', 'AN', 'STG', 'FLD')
2016 M07 ('FRN', 'AN', 'STG', 'FLD')
2016 M08 ('FRN', 'AN', 'STG', 'FLD')
2016 M09 ('FRN', 'AN', 'STG', 'FLD')
2016 M10 ('FRN', 'AN', 'STG', 'FLD')
2016 M11 ('FRN', 'AN', 'STG', 'FLD')
2016 M12 ('FRN', 'AN', 'STG', 'FLD')
2017 M01 ('FRN', 'AN', 'STG', 'FLD')
2017 M02 ('FRN', 'AN', 'STG', 'FLD')
2017 M03 ('FRN', 'AN', 'STG', 'FLD')
2017 M04 ('FRN', 'AN', 'STG', 'FLD')
2017 M05 ('FRN', 'AN', 'STG', 'FLD')
2017 M06 ('FRN', 'AN', 'STG', 'FLD')
2017 M07 ('FRN', 'AN', 'STG', 'FLD')
2017 M08 ('FRN', 'AN', 'STG', 'FLD')
2017 M09 ('FRN', 'AN', 'STG', 'FLD')
2017 M10 ('FRN', 'AN', 'STG', 'FLD')
2017 M11 ('FRN', 'AN', 'STG', 'FLD')
2017 M12 ('FRN', 'AN', 'STG', 'FLD')
2018 M01 ('FRN', 'AN', 'STG', 'FLD')
2018 M02 ('FRN', 'AN', 'STG', 'FLD')
2018 M03

C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_36596\2780167985.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.   1.   1.   1.25 1.   1.   1.   1.   1.  ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (


2016 M02 KMX


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_36596\2780167985.py:84: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.    1.    1.    1.    1.    1.25  1.25  1.    1.    1.    1.    1.
 1.    1.    1.25  1.    1.    1.    1.    1.    1.25  1.    1.    1.
 1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.25
 1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.
 1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.
 1.    1.067 1.    1.    1.    1.    1.    1.    1.    1.    1.    1.
 1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.
 1.    1.    1.    1.    1.    1.    1.25  1.    1.    1.    1.    1.
 1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.25  1.
 1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.    1.
 1.    1.    1.    1.25  1.    1.    1.    1.    1.    1.    1.    1.
 1.    1.    1.    1.    1.

2016 M03 KMX


...

2026 M01 KMX


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_36596\2780167985.py:106: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.05 1.05 1.05 ... 0.99 0.99 1.05]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (


2026 M02 KMX


C:\Users\ahmed.ali\AppData\Local\Temp\ipykernel_36596\2780167985.py:106: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.99 1.05 1.05 ... 1.05 0.99 0.99]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (



RAGU calculations completed in 619.3 seconds
Total records in all_df: 928


In [10]:
# =============================================================================
# DYNAMIC EXCEL EXPORT FUNCTIONS
# =============================================================================

def excel_cell_to_index(cell_name):
    """Converts Excel cell name (e.g., 'B2') to (row, col) indices."""
    col_str = ''.join(filter(str.isalpha, cell_name)).upper()
    col_index = 0
    for char in col_str:
        col_index = col_index * 26 + (ord(char) - ord('A')) + 1
    row_str = ''.join(filter(str.isdigit, cell_name))
    row_index = int(row_str)
    return row_index, col_index


def overwrite_values(df, start_cell, worksheet):
    """Overwrites worksheet values with DataFrame data starting at start_cell."""
    num_rows, num_cols = df.shape
    start_row, start_col = excel_cell_to_index(start_cell)
    for row in range(num_rows):
        for col in range(num_cols):
            cell = worksheet.cell(row=row + start_row, column=col + start_col)
            cell.value = df.iloc[row, col]
    return worksheet


def build_export_df(all_df_no_dropout, lob, original_model_scores, granularity, 
                    yearly_agg_years=None, recent_cutoff=2020, include_yearly_agg=True):
    """
    Build the export DataFrame for a given LOB.
    
    Parameters:
        all_df_no_dropout: Main results DataFrame
        lob: LOB to export
        original_model_scores: Original model scores for comparison
        granularity: 'q', 'm', or 'w'
        yearly_agg_years: List of years to aggregate (e.g., [2016, 2017, 2018, 2019])
        recent_cutoff: Year cutoff for individual period display
        include_yearly_agg: Whether to include yearly aggregation (False for weekly)
    
    Returns:
        DataFrame ready for Excel export
    """
    if yearly_agg_years is None:
        yearly_agg_years = DATE_CONFIG['yearly_aggregation_years']
    
    # Get base data
    result_df = all_df_no_dropout.loc[lob][
        ['vintage', 'ms_original', 'ms_gla', 'ms_exclude_ltv', 'ms_100_ltv', 'ragu_score', 'ltv']
    ].reset_index(drop=True).set_index('vintage').T.copy()
    
    # Add amount financed if available
    try:
        result_with_amt = all_df_no_dropout.loc[lob][
            ['vintage', 'ms_original', 'ms_gla', 'ms_exclude_ltv', 'ms_100_ltv', 'ragu_score', 'ltv', 'amt_financed_x']
        ].reset_index(drop=True).set_index('vintage').T.copy()
        has_amt_financed = True
    except KeyError:
        result_with_amt = result_df.copy()
        result_with_amt.loc['amt_financed_x'] = 1
        has_amt_financed = False
    
    # Merge with contract model scores
    lob_for_ms = lob if lob != 'non_kmxent' else 'non_kmxent'
    result_df = pd.concat([
        result_df, 
        original_model_scores[original_model_scores.lob == lob_for_ms].drop(
            columns=['lob', 'amt_financed']
        ).set_index(['vintage']).T.rename({'model_score': 'contract_model_score'})
    ])
    
    # Set index names
    result_df.index = ['FRN 3.1 Score', 'Gross Loss Adjustments', 'Recovery Adjustments (Excluding LTV)', 
                       'LTV Adjustments', 'Haircut on LTV Adjustments', 'LTV', 'Contract Model Score']
    
    if include_yearly_agg and has_amt_financed and granularity != 'w':
        # Build yearly aggregation
        result_yearly_df = result_with_amt.copy()
        result_yearly_df = pd.concat([
            result_yearly_df, 
            original_model_scores[original_model_scores.lob == lob_for_ms].drop(
                columns=['lob', 'amt_financed']
            ).set_index(['vintage']).T.rename({'model_score': 'contract_model_score'})
        ])
        
        for year in yearly_agg_years:
            year_str = str(year)
            if granularity == 'q':
                vintages_in_year = [f"{year_str} Q{i}" for i in range(1, 5)]
            else:
                vintages_in_year = [f"{year_str} M{str(i).zfill(2)}" for i in range(1, 13)]
            
            # Filter to existing vintages
            existing_vintages = [v for v in vintages_in_year if v in result_yearly_df.columns]
            if not existing_vintages:
                continue
            
            metric_dict = {}
            for metric in ['ms_original', 'ms_gla', 'ms_exclude_ltv', 'ms_100_ltv', 'ragu_score', 'ltv', 'amt_financed_x', 'contract_model_score']:
                try:
                    metric_dict[metric] = (
                        result_yearly_df[existing_vintages].T[metric] * result_yearly_df[existing_vintages].T.amt_financed_x
                    ).sum() / result_yearly_df[existing_vintages].T.amt_financed_x.sum()
                except KeyError:
                    pass
            result_yearly_df[year_str] = pd.Series(metric_dict)
        
        # Combine yearly and recent periods
        yearly_cols = [str(y) for y in yearly_agg_years if str(y) in result_yearly_df.columns]
        result_yearly_df = result_yearly_df[yearly_cols].drop('amt_financed_x', errors='ignore')
        result_yearly_df.index = result_df.index
        
        # Filter recent vintages
        result_df = result_df.T.reset_index()
        result_df = result_df[result_df.vintage.apply(vintage_to_float) >= recent_cutoff].set_index('vintage').T
        
        # Concatenate yearly and recent
        full_result_df = pd.concat([result_yearly_df.T, result_df.T]).T
    else:
        full_result_df = result_df.copy()
    
    # Calculate differences
    differences_df = full_result_df.copy()
    differences_df.loc['FRN 3.1 Score'] = full_result_df.loc['FRN 3.1 Score'] - full_result_df.loc['Contract Model Score']
    differences_df.loc['Gross Loss Adjustments'] = full_result_df.loc['Gross Loss Adjustments'] - full_result_df.loc['FRN 3.1 Score']
    differences_df.loc['Recovery Adjustments (Excluding LTV)'] = full_result_df.loc['Recovery Adjustments (Excluding LTV)'] - full_result_df.loc['Gross Loss Adjustments']
    differences_df.loc['LTV Adjustments'] = full_result_df.loc['LTV Adjustments'] - full_result_df.loc['Recovery Adjustments (Excluding LTV)']
    differences_df.loc['Haircut on LTV Adjustments'] = full_result_df.loc['Haircut on LTV Adjustments'] - full_result_df.loc['LTV Adjustments']
    differences_df.loc['RAGU Score'] = full_result_df.loc['LTV Adjustments'].copy()
    differences_df.loc['RAGU Score (Adjusted)'] = full_result_df.loc['Haircut on LTV Adjustments'].copy()
    
    # Select output columns
    xlsx_df = differences_df.T[['Contract Model Score', 'FRN 3.1 Score', 'Gross Loss Adjustments', 
                                'Recovery Adjustments (Excluding LTV)', 'LTV Adjustments', 'RAGU Score', 'LTV']].T
    
    return xlsx_df


def export_to_excel(all_df_no_dropout, original_model_scores, granularity, 
                    output_filename=None, template_filename=None):
    """
    Export RAGU results to Excel using a template workbook.
    
    Parameters:
        all_df_no_dropout: Main results DataFrame
        original_model_scores: Original model scores
        granularity: 'q', 'm', or 'w'
        output_filename: Output file name (auto-generated if None)
        template_filename: Template file name (auto-selected based on granularity if None)
    """
    # Select template and output files
    if template_filename is None:
        template_filename = 'weekly RAGU newrecovery.xlsx' if granularity == 'w' else 'new_recovery_ragu.xlsx'
    
    if output_filename is None:
        output_filename = 'weekly RAGU newrecovery.xlsx' if granularity == 'w' else 'cursor_ragu.xlsx'
    
    # Load workbook
    print(f"Loading template: {template_filename}")
    workbook = openpyxl.load_workbook(template_filename)
    
    # Get cell positions for this mode
    cell_config = EXCEL_CELL_CONFIG['weekly'] if granularity == 'w' else EXCEL_CELL_CONFIG['quarterly_monthly']
    
    # Get worksheet
    worksheet = workbook[f'Data Tables ({granularity.upper()})']
    
    # Determine if we include yearly aggregation (only for quarterly/monthly)
    include_yearly = granularity != 'w'
    
    # Export NonKMXENT LOBs (AN, FRN, STG, non_kmxent)
    for lob in ['AN', 'FRN', 'STG', 'non_kmxent']:
        try:
            xlsx_df = build_export_df(all_df_no_dropout, lob, original_model_scores, 
                                      granularity, include_yearly_agg=include_yearly)
            worksheet = overwrite_values(xlsx_df, cell_config[lob], worksheet)
            print(f"  Exported {lob} to {cell_config[lob]}")
        except Exception as e:
            print(f"  Warning: Could not export {lob}: {e}")
    
    # Export ENT, FLD (no yearly aggregation)
    for lob in ['ENT', 'FLD']:
        try:
            xlsx_df = build_export_df(all_df_no_dropout, lob, original_model_scores, 
                                      granularity, include_yearly_agg=False)
            worksheet = overwrite_values(xlsx_df, cell_config[lob], worksheet)
            print(f"  Exported {lob} to {cell_config[lob]}")
        except Exception as e:
            print(f"  Warning: Could not export {lob}: {e}")
    
    # Export KMX
    try:
        xlsx_df = build_export_df(all_df_no_dropout, 'KMX', original_model_scores, 
                                  granularity, include_yearly_agg=include_yearly)
        worksheet = overwrite_values(xlsx_df, cell_config['KMX'], worksheet)
        print(f"  Exported KMX to {cell_config['KMX']}")
    except Exception as e:
        print(f"  Warning: Could not export KMX: {e}")
    
    # Export POS
    try:
        xlsx_df = build_export_df(all_df_no_dropout, 'POS', original_model_scores, 
                                  granularity, include_yearly_agg=include_yearly)
        worksheet = overwrite_values(xlsx_df, cell_config['POS'], worksheet)
        print(f"  Exported POS to {cell_config['POS']}")
    except Exception as e:
        print(f"  Warning: Could not export POS: {e}")
    
    # Save workbook
    workbook.save(output_filename)
    print(f"\nSaved to: {output_filename}")
    
    return output_filename

In [11]:
# =============================================================================
# EXECUTE EXCEL EXPORT
# =============================================================================

# Export results to Excel
print("Exporting to Excel...")
output_file = export_to_excel(all_df_no_dropout, original_model_scores, granularity)

# Optionally save all_df to CSV for review
all_df.to_csv('all_df.csv')
print("\nAlso saved all_df.csv for review")

Exporting to Excel...
Loading template: new_recovery_ragu.xlsx
  Exported AN to B2
  Exported FRN to B20
  Exported STG to B29
  Exported non_kmxent to B47
  Exported ENT to F38
  Exported FLD to F11
  Exported KMX to B56
  Exported POS to B65

Saved to: cursor_ragu.xlsx

Also saved all_df.csv for review


In [12]:
# =============================================================================
# LEAVE-OUT ANALYSIS (Optional)
# =============================================================================
# Build leave-out analysis DataFrame for examining factor impacts

def build_leave_out_analysis(full_df_list_dict_dict, leave_out_list, lob_type='non_kmxent'):
    """Build leave-out analysis DataFrame showing RAGU impact per factor."""
    all_leave_out_df = pd.DataFrame()
    
    for leave_out in leave_out_list:
        leave_out_df = pd.DataFrame()
        for full_df in full_df_list_dict_dict[leave_out][lob_type]:
            if lob_type in full_df.index:
                leave_out_df = pd.concat([leave_out_df, pd.DataFrame(full_df.loc[lob_type]).T])
        
        if not leave_out_df.empty:
            leave_out_df['leave_out'] = leave_out
            leave_out_df = leave_out_df.reset_index().set_index('vintage')
            all_leave_out_df = pd.concat([all_leave_out_df, leave_out_df])
    
    if not all_leave_out_df.empty:
        all_leave_out_df = all_leave_out_df.reset_index().set_index('leave_out')
    
    return all_leave_out_df


# Only run if full leave-out analysis was performed
if len(leave_out_list) > 1:
    all_leave_out_df = build_leave_out_analysis(full_df_list_dict_dict, leave_out_list)
    
    # Calculate RAGU differences
    leave_out_results = {}
    for leave_out in leave_out_list:
        ragu_diff = np.abs(
            all_leave_out_df.loc['None'].reset_index().ragu_score - 
            all_leave_out_df.loc[leave_out].reset_index().ragu_score
        ).mean()
        leave_out_results[leave_out] = ragu_diff
    
    # Display results
    results_df = pd.Series(leave_out_results).reset_index()
    results_df.columns = ['Factor', 'RAGU_Impact']
    print("\nLeave-Out Analysis Results (sorted by impact):")
    print(results_df.sort_values('RAGU_Impact', ascending=False).to_string(index=False))
else:
    print("Leave-out analysis not performed (only 'None' in leave_out_list)")

Leave-out analysis not performed (only 'None' in leave_out_list)


In [13]:
# =============================================================================
# SUMMARY
# =============================================================================

print("=" * 60)
print("RAGU SCORE ANALYSIS - SUMMARY")
print("=" * 60)
print(f"Granularity: {granularity.upper()} ({'Quarterly' if granularity == 'q' else 'Monthly' if granularity == 'm' else 'Weekly'})")
print(f"Date column: {date_col}")
print(f"Start year: {start_year}")
print(f"Records processed: {len(all_df):,}")
print(f"Output file: {output_file if 'output_file' in dir() else 'N/A'}")
print("=" * 60)

# Display sample results
print("\nSample of all_df (first 10 rows):")
all_df.head(10)

RAGU SCORE ANALYSIS - SUMMARY
Granularity: M (Monthly)
Date column: book_date
Start year: 2016
Records processed: 928
Output file: cursor_ragu.xlsx

Sample of all_df (first 10 rows):


,amt_financed_x,loss_multiplier,recovery_unadjusted_multiplier,ltv,bbvalue,ltv_realization_factor,recovery_100_ltv_multiplier,recovery_multiplier,vintage,model_score,amt_financed_y,est_unit_loss,unit_loss_score,ms_original,baselined_recovery,baselined_unadjusted_recovery,baselined_100_ltv_recovery,ms_gla,ms_exclude_ltv,only_recovery_ragu,ms_100_ltv,ragu_score
lob,,,,,,,,,,,,,,,,,,,,,,
AN,9145187.23,0.963058,0.550272,2.009341,9601.722534,0.0,0.0,0.0,2016 M01,129.760371,7781530.53,0.5,130.683928,129.760371,0.0,1.000494,0.0,130.683928,130.701703,0.017775,130.683928,130.683928
FLD,555634.69,0.991388,0.540342,1.904410,9349.497930,0.0,0.0,0.0,2016 M01,136.535021,76648.00,0.5,136.750318,136.535021,0.0,0.982439,0.0,136.750318,136.101523,-0.648795,136.750318,136.750318
FRN,5665833.98,1.054231,0.541868,2.226625,8314.750108,0.0,0.0,0.0,2016 M01,128.589400,3254221.42,0.5,127.233619,128.589400,0.0,0.985214,0.0,127.233619,126.723933,-0.509686,127.233619,127.233619
STG,8856093.24,0.970145,0.552043,2.002217,9423.348521,0.0,0.0,0.0,2016 M01,129.003442,8187381.84,0.5,129.749823,129.003442,0.0,1.003714,0.0,129.749823,129.882831,0.133008,129.749823,129.749823
non_kmxent,24222749.14,NaN,0.548726,2.055154,NaN,NaN,0.0,0.0,2016 M01,NaN,NaN,0.5,129.674517,129.365134,0.0,0.997683,0.0,129.674517,129.592092,-0.082425,129.674517,129.674517
AN,13095266.56,0.970719,0.541726,1.903139,9321.588847,0.0,0.0,0.0,2016 M02,128.308058,11147051.92,0.5,129.040081,128.308058,0.0,0.984957,0.0,129.040081,128.514290,-0.525792,129.040081,129.040081
FLD,567606.57,1.016056,0.526919,1.847937,8636.288674,0.0,0.0,0.0,2016 M02,134.510238,159067.95,0.5,134.108835,134.510238,0.0,0.958035,0.0,134.108835,132.626119,-1.482716,134.108835,134.108835
FRN,7040731.72,1.015945,0.548287,2.146608,8446.186071,0.0,0.0,0.0,2016 M02,128.505771,4408072.04,0.5,128.107149,128.505771,0.0,0.996885,0.0,128.107149,127.997757,-0.109392,128.107149,128.107149
STG,12914234.93,0.961861,0.543322,1.981074,8928.456714,0.0,0.0,0.0,2016 M02,127.693263,11926357.38,0.5,128.646727,127.693263,0.0,0.987858,0.0,128.646727,128.222386,-0.424341,128.646727,128.646727
